# Lab 1.3: Structured Output via `tool_use`

You are building an invoice extraction pipeline for a finance team. Invoices arrive from different vendors -- some include warranty info, some don't, some have ambiguous payment status.

**What you will learn:**
- Why prompt-based JSON extraction is unreliable in production
- Why `tool_choice: "auto"` still lets the model dodge structured output
- Why required fields without nullable types cause the model to **fabricate** data
- How to design schemas with nullable fields, enum + "other", and "unclear" escape valves
- How `tool_choice: "any"` guarantees structured output

**The pattern:** Prompt-only extraction fails in three ways: inconsistent format, optional tool calls, and forced fabrication. Schema design with escape valves fixes all three.

## Setup

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var
MODEL = "claude-sonnet-4-20250514"

## Test Invoices

Three invoices that cover the critical edge cases: a complete invoice, one missing warranty info, and one with ambiguous payment status.

In [ ]:
INVOICE_FULL = """
INVOICE #2024-0892
Vendor: TechParts International
Date: 2024-03-15

Bill To: Acme Corp, 123 Main St, Springfield, IL 62701
Ship To: Acme Corp Warehouse, 456 Industrial Pkwy, Springfield, IL 62704

Items:
  1. Widget Pro X200 -- Qty: 10 -- Unit: $45.00 -- Total: $450.00
  2. Connector Kit C-50 -- Qty: 5 -- Unit: $22.00 -- Total: $110.00
  3. Mounting Bracket MB-L -- Qty: 20 -- Unit: $8.50 -- Total: $170.00

Subtotal: $730.00
Tax (8%): $58.40
Shipping: $25.00
Total: $813.40

Payment Status: Paid (Wire transfer, 2024-03-18)
Warranty: All items covered under 2-year manufacturer warranty.
Warranty Expiry: 2026-03-15
"""

INVOICE_NO_WARRANTY = """
INVOICE #2024-1045
Vendor: QuickSupply Co
Date: 2024-06-01

Bill To: Acme Corp, 123 Main St, Springfield, IL 62701

Items:
  1. Paper Ream (A4, 500 sheets) -- Qty: 50 -- Unit: $4.99 -- Total: $249.50
  2. Ink Cartridge BK-200 -- Qty: 10 -- Unit: $18.00 -- Total: $180.00

Subtotal: $429.50
Tax (8%): $34.36
Total: $463.86

Payment Status: Paid (Credit card ending 4521, 2024-06-01)
"""

INVOICE_AMBIGUOUS_PAYMENT = """
MEMO / INVOICE #PRE-2024-003
Vendor: Consolidated Services LLC
Date: 2024-08-22

To: Acme Corp

Service: Annual maintenance contract renewal
Amount: $12,000.00

Note: Invoice pending final approval from procurement. Deposit of $4,000
was received on 2024-08-10. Remaining balance to be settled upon contract
signing. Status under review.
"""

print("Three test invoices loaded:")
print(f"  1. Full invoice (all fields present)")
print(f"  2. No warranty (warranty info absent)")
print(f"  3. Ambiguous payment (partially paid, status unclear)")

---
## Phase 1: THE WRONG WAY

Three increasingly common approaches that all fail in production. We will run each one and measure the failure modes.

### Wrong Way #1: Prompt-based JSON extraction (no tool_use)

The most common first attempt -- just ask the model to return JSON in its text response.

In [ ]:
def extract_with_prompt(invoice_text):
    """Naive approach: ask the model to return JSON in its text response."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        messages=[{"role": "user", "content": f"""Extract invoice data as JSON with these fields:
- invoice_number, vendor_name, date, line_items, subtotal, tax, shipping, total,
  payment_status, warranty_info, warranty_expiry, billing_address, shipping_address

Return ONLY valid JSON, no other text.

{invoice_text}"""}]
    )
    return response.content[0].text


# Run against all three invoices
print("=" * 60)
print("PROMPT-BASED EXTRACTION (no tool_use)")
print("=" * 60)

for name, invoice in [("Full", INVOICE_FULL), ("No Warranty", INVOICE_NO_WARRANTY), ("Ambiguous", INVOICE_AMBIGUOUS_PAYMENT)]:
    raw = extract_with_prompt(invoice)
    print(f"\n--- {name} Invoice ---")
    
    # Check: is it valid JSON?
    clean = raw.strip()
    if clean.startswith("```"):
        print("  FORMAT ISSUE: Response wrapped in markdown code fences")
        clean = clean.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    
    try:
        data = json.loads(clean)
        print(f"  Valid JSON: yes")
        print(f"  Fields returned: {list(data.keys())}")
        
        # Check warranty field for no-warranty invoice
        if name == "No Warranty":
            warranty = data.get("warranty_info") or data.get("warranty_expiry") or data.get("warranty")
            if warranty and warranty not in ("N/A", "None", "null", None, ""):
                print(f"  WARNING: warranty field = {warranty!r} -- possible fabrication!")
            else:
                print(f"  warranty handling: {warranty!r}")
    except json.JSONDecodeError as e:
        print(f"  Valid JSON: NO -- {e}")
        print(f"  Raw response (first 200 chars): {raw[:200]}")

**Observation:** Even when you say "Return ONLY valid JSON", the model sometimes wraps it in markdown code fences (` ```json ... ``` `), uses inconsistent field names, or represents missing data in unpredictable ways ("N/A", empty string, omitting the key entirely). Every downstream parser must handle all these variants.

### Wrong Way #2: tool_choice "auto" -- the model can dodge

A common next step: define a tool for extraction but leave `tool_choice` on `"auto"`. The model *can* call the tool, but it can also just return text.

In [ ]:
# A basic extraction tool
basic_tool = {
    "name": "extract_invoice",
    "description": "Extract structured data from an invoice document.",
    "input_schema": {
        "type": "object",
        "required": ["invoice_number", "vendor_name", "total"],
        "properties": {
            "invoice_number": {"type": "string"},
            "vendor_name": {"type": "string"},
            "date": {"type": "string"},
            "total": {"type": "number"},
            "payment_status": {"type": "string"},
            "warranty_expiry": {"type": "string"},
        }
    }
}


def extract_with_auto(invoice_text):
    """Use tool_choice='auto' -- the model MIGHT use the tool."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        tools=[basic_tool],
        tool_choice={"type": "auto"},
        messages=[{"role": "user", "content": f"Extract the invoice data:\n{invoice_text}"}]
    )
    # Check what we got back
    tool_blocks = [b for b in response.content if b.type == "tool_use"]
    text_blocks = [b for b in response.content if b.type == "text"]
    return {"tool_blocks": tool_blocks, "text_blocks": text_blocks}


print("=" * 60)
print("tool_choice='auto' -- Running 5 times on ambiguous invoice")
print("=" * 60)

tool_call_count = 0
text_only_count = 0

for run in range(5):
    result = extract_with_auto(INVOICE_AMBIGUOUS_PAYMENT)
    used_tool = len(result["tool_blocks"]) > 0
    if used_tool:
        tool_call_count += 1
        print(f"  Run {run + 1}: Tool call -- structured output")
    else:
        text_only_count += 1
        text = result["text_blocks"][0].text[:100] if result["text_blocks"] else "(empty)"
        print(f"  Run {run + 1}: TEXT response -- '{text}...'")

print(f"\nResults: {tool_call_count}/5 tool calls, {text_only_count}/5 text responses")
if text_only_count > 0:
    print(f"\nPROBLEM: {text_only_count} times the model returned text instead of structured data.")
    print("Your downstream pipeline expects a dict -- it gets a string. This crashes in production.")

**Observation:** With `tool_choice: "auto"`, the model treats the tool as optional. For ambiguous documents especially, it may decide that a text explanation is more appropriate than a structured extraction. In a pipeline, this means unpredictable response types.

### Wrong Way #3: All fields required, no nullable types -- forced fabrication

This is the most dangerous failure mode. When your schema says a field is required and its type is `string` (not nullable), the model MUST provide a value. If the information is not in the document, the model **fabricates** one.

In [ ]:
# Schema where warranty_expiry is REQUIRED and NOT nullable
fabrication_tool = {
    "name": "extract_invoice",
    "description": "Extract structured data from an invoice document.",
    "input_schema": {
        "type": "object",
        "required": ["invoice_number", "vendor_name", "total", "warranty_expiry", "payment_status"],
        "properties": {
            "invoice_number": {"type": "string"},
            "vendor_name": {"type": "string"},
            "total": {"type": "number"},
            "warranty_expiry": {"type": "string", "description": "Warranty expiration date"},
            "payment_status": {
                "type": "string",
                "enum": ["paid", "unpaid"]
            }
        }
    }
}


def extract_forced(invoice_text, tool):
    """Force the model to call the tool."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        tools=[tool],
        tool_choice={"type": "tool", "name": "extract_invoice"},
        messages=[{"role": "user", "content": f"Extract invoice data:\n{invoice_text}"}]
    )
    for block in response.content:
        if block.type == "tool_use":
            return block.input
    return None


print("=" * 60)
print("FORCED FABRICATION: Required non-nullable warranty_expiry")
print("=" * 60)

print("\n--- No Warranty Invoice (warranty info is ABSENT from document) ---")
fabrication_count = 0
results = []

for run in range(5):
    result = extract_forced(INVOICE_NO_WARRANTY, fabrication_tool)
    warranty = result.get("warranty_expiry", "MISSING")
    results.append(warranty)
    is_fabricated = warranty not in (None, "null", "", "N/A", "MISSING")
    if is_fabricated:
        fabrication_count += 1
    print(f"  Run {run + 1}: warranty_expiry = {warranty!r}  {'<-- FABRICATED' if is_fabricated else ''}")

print(f"\nFabrication rate: {fabrication_count}/5 ({fabrication_count * 20}%)")
print("\nThe model invented warranty dates that DO NOT EXIST in the document.")
print("This is not a hallucination bug -- it is a SCHEMA DESIGN failure.")
print("The schema demanded a string value, so the model provided one.")

print("\n--- Ambiguous Payment Invoice (status is genuinely unclear) ---")
for run in range(3):
    result = extract_forced(INVOICE_AMBIGUOUS_PAYMENT, fabrication_tool)
    status = result.get("payment_status")
    print(f"  Run {run + 1}: payment_status = {status!r}")

print("\nWith only 'paid' and 'unpaid' as enum options, the model is forced to pick one.")
print("The real answer is 'partial/unclear' -- but the schema does not allow it.")

### Phase 1 Summary

| Failure Mode | Root Cause | Consequence |
|---|---|---|
| Inconsistent JSON format | Prompt-based extraction, no schema enforcement | Parser breakage, markdown wrapping |
| Model returns text instead of tool call | `tool_choice: "auto"` lets model dodge | Pipeline crashes on unexpected type |
| Model fabricates missing data | Required non-nullable fields for optional data | Silent data corruption -- the worst kind |

---
## Phase 2: THE RIGHT WAY

Design the schema to match the reality of the data: some fields are optional, some categories are open-ended, and sometimes the answer is genuinely "unclear".

### The correct extraction tool

Three schema design principles:
1. **Nullable fields** for data that may be absent: `"type": ["string", "null"]`
2. **Enum + "other"** for categories that might not fit predefined options
3. **"unclear" escape valve** for genuinely ambiguous data

In [ ]:
correct_tool = {
    "name": "extract_invoice",
    "description": "Extract structured data from an invoice document. Use null for fields where the information is not present in the document. Use 'unclear' for ambiguous payment status. Use 'other' for document types that do not fit standard categories.",
    "input_schema": {
        "type": "object",
        "required": [
            "invoice_number", "vendor_name", "purchase_date",
            "document_type", "document_type_detail",
            "line_items", "subtotal", "tax", "shipping", "total",
            "payment_status",
            "warranty_info", "warranty_expiry",
            "billing_address", "shipping_address"
        ],
        "properties": {
            "invoice_number": {
                "type": "string",
                "description": "The invoice number or identifier"
            },
            "vendor_name": {
                "type": "string",
                "description": "Name of the vendor/supplier"
            },
            "purchase_date": {
                "type": "string",
                "description": "Date in ISO 8601 format (YYYY-MM-DD)"
            },
            "document_type": {
                "type": "string",
                "enum": ["invoice", "credit_note", "purchase_order", "receipt", "other"],
                "description": "Type of document. Use 'other' if it does not fit standard categories."
            },
            "document_type_detail": {
                "type": ["string", "null"],
                "description": "If document_type is 'other', describe the type here. Null otherwise."
            },
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "quantity": {"type": "number"},
                        "unit_price": {"type": "number"},
                        "line_total": {"type": "number"}
                    }
                },
                "description": "List of line items. Empty array if no itemized list."
            },
            "subtotal": {
                "type": ["number", "null"],
                "description": "Subtotal before tax/shipping. Null if not stated."
            },
            "tax": {
                "type": ["number", "null"],
                "description": "Tax amount. Null if not stated."
            },
            "shipping": {
                "type": ["number", "null"],
                "description": "Shipping cost. Null if not stated."
            },
            "total": {
                "type": "number",
                "description": "Total amount due or paid"
            },
            "payment_status": {
                "type": "string",
                "enum": ["paid", "unpaid", "partial", "unclear"],
                "description": "Payment status. Use 'unclear' when the status is ambiguous or under review."
            },
            "warranty_info": {
                "type": ["string", "null"],
                "description": "Warranty details. Null if no warranty information in the document."
            },
            "warranty_expiry": {
                "type": ["string", "null"],
                "description": "Warranty expiration date (YYYY-MM-DD). Null if no warranty information."
            },
            "billing_address": {
                "type": ["string", "null"],
                "description": "Billing address. Null if not stated."
            },
            "shipping_address": {
                "type": ["string", "null"],
                "description": "Shipping address. Null if not stated."
            }
        }
    }
}

print("Correct tool defined.")
print(f"  Nullable fields: warranty_info, warranty_expiry, shipping_address, billing_address, subtotal, tax, shipping, document_type_detail")
print(f"  Enum with 'other': document_type")
print(f"  Enum with 'unclear': payment_status")

### Extract with `tool_choice: "any"`

We use `tool_choice: {"type": "any"}` (or `{"type": "tool", "name": "extract_invoice"}` for a single tool) to **guarantee** the model returns structured output. No text responses, no dodging.

In [ ]:
def extract_correct(invoice_text):
    """Extract using correct schema + forced tool call."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=4096,
        tools=[correct_tool],
        tool_choice={"type": "any"},
        messages=[{"role": "user", "content": f"Extract all invoice data from this document:\n{invoice_text}"}]
    )
    for block in response.content:
        if block.type == "tool_use":
            return block.input
    return None


print("=" * 60)
print("CORRECT EXTRACTION: nullable fields + forced tool call")
print("=" * 60)

for name, invoice in [("Full", INVOICE_FULL), ("No Warranty", INVOICE_NO_WARRANTY), ("Ambiguous", INVOICE_AMBIGUOUS_PAYMENT)]:
    result = extract_correct(invoice)
    print(f"\n--- {name} Invoice ---")
    print(json.dumps(result, indent=2))

### Verify: no fabrication, correct handling of ambiguity

In [ ]:
print("=" * 60)
print("VERIFICATION: Run 5 times, check fabrication rate")
print("=" * 60)

# Test no-warranty invoice
print("\n--- No Warranty Invoice: warranty_expiry should always be null ---")
fabrication_count_correct = 0
for run in range(5):
    result = extract_correct(INVOICE_NO_WARRANTY)
    warranty = result.get("warranty_expiry")
    warranty_info = result.get("warranty_info")
    fabricated = warranty is not None
    if fabricated:
        fabrication_count_correct += 1
    print(f"  Run {run + 1}: warranty_expiry={warranty!r}, warranty_info={warranty_info!r}  {'<-- FABRICATED' if fabricated else 'OK'}")

print(f"\nFabrication rate (correct schema): {fabrication_count_correct}/5 ({fabrication_count_correct * 20}%)")

# Test ambiguous payment
print("\n--- Ambiguous Payment Invoice: payment_status should be 'unclear' or 'partial' ---")
for run in range(5):
    result = extract_correct(INVOICE_AMBIGUOUS_PAYMENT)
    status = result.get("payment_status")
    correct = status in ("unclear", "partial")
    print(f"  Run {run + 1}: payment_status={status!r}  {'OK' if correct else '<-- WRONG'}")

### Phase 2 Comparison: Wrong vs Right

In [ ]:
print("=" * 60)
print("COMPARISON: Phase 1 (Wrong) vs Phase 2 (Right)")
print("=" * 60)
print()
print(f"{'Metric':<45} {'Wrong':>10} {'Right':>10}")
print(f"{'-'*45} {'-'*10} {'-'*10}")
print(f"{'Guaranteed structured output':.<45} {'No':>10} {'Yes':>10}")
print(f"{'Consistent response type':.<45} {'No':>10} {'Yes':>10}")
print(f"{'Fabrication rate (warranty on no-warranty)':.<45} {'~60-100%':>10} {'0%':>10}")
print(f"{'Handles ambiguous payment':.<45} {'Forced':>10} {'Correct':>10}")
print(f"{'Missing data represented as':.<45} {'Varies':>10} {'null':>10}")
print()
print("Key design principles:")
print("  1. Nullable fields: type ['string', 'null'] for data that may be absent")
print("  2. Enum + 'other': escape valve for unexpected categories")
print("  3. 'unclear' status: escape valve for genuinely ambiguous data")
print("  4. tool_choice 'any': guarantees the model uses the schema")
print("  5. Descriptions with null guidance: 'Null if not stated in the document'")

### tool_choice reference

| `tool_choice` | When to use | Behavior |
|---|---|---|
| `{"type": "auto"}` | Model decides whether extraction applies at all | Model may return text instead of tool call |
| `{"type": "any"}` | Multiple tools, model must pick one | Guarantees a tool call, model picks which |
| `{"type": "tool", "name": "..."}` | Single extraction schema, always call it | Guarantees that specific tool is called |

---
## Phase 3: YOUR TURN

**Scenario:** Design an extraction schema for a product catalog. Products have varying attributes -- some have physical dimensions, some do not. Categories should use a predefined enum with an "other" escape valve.

### Requirements
1. Define a tool named `"extract_product"` with these fields:
   - `product_name` (string, required)
   - `sku` (string, required)
   - `category` (enum: "electronics", "furniture", "clothing", "food", "other" -- required)
   - `category_detail` (string or null -- if category is "other", describe it here)
   - `price` (number, required)
   - `weight_kg` (number or null -- some products do not list weight)
   - `dimensions_cm` (object with length/width/height, or null -- some products have no dimensions)
   - `in_stock` (boolean, required)
   - `availability_status` (enum: "available", "backorder", "discontinued", "unclear" -- required)
   - `warranty_months` (integer or null -- not all products have warranties)

2. Choose the correct `tool_choice` for: extracting product data when this is the only tool.

3. Test it against the product listings below.

In [ ]:
PRODUCT_WITH_DIMENSIONS = """
Product: ErgoDesk Pro Standing Desk
SKU: FURN-ED-4872
Category: Office Furniture
Price: $549.99
Weight: 32.5 kg
Dimensions: 120cm x 60cm x 75-125cm (adjustable height)
In Stock: Yes (14 units)
Warranty: 5 years
"""

PRODUCT_NO_DIMENSIONS = """
Product: Premium Bluetooth Earbuds
SKU: ELEC-BT-2200
Category: Electronics / Audio
Price: $79.99
Weight: 0.045 kg
In Stock: Yes
Warranty: 1 year manufacturer warranty
"""

PRODUCT_UNUSUAL_CATEGORY = """
Product: Artisanal Beeswax Candle Set (6-pack)
SKU: CRAFT-BWC-006
Category: Home & Craft
Price: $34.50
In Stock: Limited availability -- check back next week for restock.
"""

print("Three product listings loaded.")
print("  1. Standing desk -- has dimensions, weight, warranty")
print("  2. Earbuds -- no dimensions, has weight and warranty")
print("  3. Candle set -- unusual category, no weight/dimensions/warranty, ambiguous stock")

In [ ]:
# ============================================================
# YOUR TURN: Define the product extraction tool
# ============================================================

product_tool = {
    "name": "extract_product",
    "description": "TODO: Write a description that guides the model on null handling",
    "input_schema": {
        "type": "object",
        "required": [],  # TODO: List required fields
        "properties": {
            # TODO: Define each field with correct types
            # Remember:
            #   - Nullable fields: "type": ["string", "null"]
            #   - Enum with escape valve: "enum": ["opt1", "opt2", "other"]
            #   - Nested object or null: use anyOf or type array
        }
    }
}

# TODO: Set the correct tool_choice for single-schema extraction
product_tool_choice = {"type": "TODO"}  # What goes here?

print("Define your product_tool and product_tool_choice above, then run the next cell.")

In [ ]:
# ============================================================
# CHECKER: Validate your schema design
# ============================================================

def check_product_schema(tool, tool_choice):
    """Validate the product extraction schema."""
    errors = []
    
    # Check tool structure
    if tool.get("name") != "extract_product":
        errors.append("Tool name must be 'extract_product'")
    
    schema = tool.get("input_schema", {})
    props = schema.get("properties", {})
    required = schema.get("required", [])
    
    # Check required fields are listed
    for field in ["product_name", "sku", "category", "price", "in_stock", "availability_status"]:
        if field not in required:
            errors.append(f"'{field}' should be in required list")
    
    # Check nullable fields
    nullable_fields = ["weight_kg", "dimensions_cm", "warranty_months", "category_detail"]
    for field in nullable_fields:
        if field not in props:
            errors.append(f"Missing field: '{field}'")
            continue
        field_type = props[field].get("type")
        any_of = props[field].get("anyOf")
        is_nullable = False
        if isinstance(field_type, list) and "null" in field_type:
            is_nullable = True
        if any_of and any(t.get("type") == "null" for t in any_of):
            is_nullable = True
        if not is_nullable:
            errors.append(f"'{field}' must be nullable (include 'null' in type)")
    
    # Check enum with 'other'
    if "category" in props:
        cat_enum = props["category"].get("enum", [])
        if "other" not in cat_enum:
            errors.append("'category' enum must include 'other'")
    else:
        errors.append("Missing field: 'category'")
    
    # Check enum with 'unclear'
    if "availability_status" in props:
        avail_enum = props["availability_status"].get("enum", [])
        if "unclear" not in avail_enum:
            errors.append("'availability_status' enum must include 'unclear'")
    else:
        errors.append("Missing field: 'availability_status'")
    
    # Check tool_choice
    if tool_choice.get("type") not in ("tool", "any"):
        errors.append(f"tool_choice should force tool usage. Got type='{tool_choice.get('type')}'")
    
    # Report
    if errors:
        print("SCHEMA CHECK FAILED:")
        for e in errors:
            print(f"  - {e}")
        return False
    else:
        print("SCHEMA CHECK PASSED -- all design principles applied correctly.")
        return True


schema_ok = check_product_schema(product_tool, product_tool_choice)

if schema_ok:
    print("\nNow test extraction against the three products...")

In [ ]:
# ============================================================
# TEST: Extract from all three products (run after schema passes)
# ============================================================

if schema_ok:
    for name, product_text in [
        ("Standing Desk (has dimensions)", PRODUCT_WITH_DIMENSIONS),
        ("Earbuds (no dimensions)", PRODUCT_NO_DIMENSIONS),
        ("Candle Set (unusual category, ambiguous stock)", PRODUCT_UNUSUAL_CATEGORY),
    ]:
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            tools=[product_tool],
            tool_choice=product_tool_choice,
            messages=[{"role": "user", "content": f"Extract product data:\n{product_text}"}]
        )
        for block in response.content:
            if block.type == "tool_use":
                result = block.input
                break
        
        print(f"\n--- {name} ---")
        print(json.dumps(result, indent=2))
        
        # Quick checks
        if "no dimensions" in name.lower():
            dims = result.get("dimensions_cm")
            if dims is not None:
                print(f"  WARNING: dimensions_cm should be null, got {dims!r}")
            else:
                print(f"  OK: dimensions_cm is null (correct)")
        
        if "unusual category" in name.lower():
            cat = result.get("category")
            avail = result.get("availability_status")
            warranty = result.get("warranty_months")
            print(f"  category={cat!r} (should be 'other')")
            print(f"  availability_status={avail!r} (should be 'unclear' or 'backorder')")
            print(f"  warranty_months={warranty!r} (should be null)")
else:
    print("Fix the schema errors above first.")

---
## Phase 4: STRESS TEST

Run the correct extraction tool against 5 diverse invoices, including edge cases. Verify that nullable fields return null (not fabricated values) and ambiguous data gets the correct escape-valve value.

In [ ]:
STRESS_INVOICES = {
    "Complete (all fields)": INVOICE_FULL,

    "Missing warranty + shipping": INVOICE_NO_WARRANTY,

    "Ambiguous payment": INVOICE_AMBIGUOUS_PAYMENT,

    "Minimal (barely an invoice)": """
Quick receipt
From: Joe's Fix-It Shop
Date: Jan 5 2024
Repair service: $150
Paid cash.
""",

    "Non-standard format (email)": """
Subject: Payment confirmation - Project Alpha
From: billing@globalconsulting.com
Date: 2024-11-01

Hi,

This confirms your payment of $45,000 for Project Alpha consulting services
(Invoice #GC-2024-1101). Payment was received via wire transfer on Oct 30.

No warranty applicable for consulting services.

Thanks,
Global Consulting Billing Team
"""
}

print(f"Stress testing {len(STRESS_INVOICES)} diverse documents...\n")

stress_results = {}
for name, doc in STRESS_INVOICES.items():
    result = extract_correct(doc)
    stress_results[name] = result
    print(f"--- {name} ---")
    print(json.dumps(result, indent=2))
    print()

In [ ]:
# ============================================================
# STRESS TEST VALIDATION
# ============================================================

print("=" * 60)
print("STRESS TEST RESULTS")
print("=" * 60)

all_passed = True

# Check 1: No warranty invoice -- warranty fields must be null
no_warranty = stress_results.get("Missing warranty + shipping", {})
if no_warranty.get("warranty_expiry") is not None:
    print("FAIL: 'Missing warranty' invoice has non-null warranty_expiry (fabrication)")
    all_passed = False
else:
    print("PASS: 'Missing warranty' invoice -> warranty_expiry is null")

if no_warranty.get("warranty_info") is not None:
    print("FAIL: 'Missing warranty' invoice has non-null warranty_info (fabrication)")
    all_passed = False
else:
    print("PASS: 'Missing warranty' invoice -> warranty_info is null")

# Check 2: Ambiguous payment -- must be 'unclear' or 'partial'
ambiguous = stress_results.get("Ambiguous payment", {})
if ambiguous.get("payment_status") not in ("unclear", "partial"):
    print(f"FAIL: Ambiguous payment -> status should be 'unclear' or 'partial', got {ambiguous.get('payment_status')!r}")
    all_passed = False
else:
    print(f"PASS: Ambiguous payment -> payment_status={ambiguous.get('payment_status')!r}")

# Check 3: Minimal invoice -- many fields should be null
minimal = stress_results.get("Minimal (barely an invoice)", {})
for field in ["warranty_expiry", "warranty_info", "shipping_address", "shipping"]:
    if minimal.get(field) is not None:
        print(f"FAIL: Minimal invoice -> '{field}' should be null, got {minimal.get(field)!r}")
        all_passed = False
    else:
        print(f"PASS: Minimal invoice -> '{field}' is null")

# Check 4: Email invoice -- structured extraction still works
email_inv = stress_results.get("Non-standard format (email)", {})
if not email_inv.get("invoice_number"):
    print("FAIL: Email invoice -> missing invoice_number")
    all_passed = False
else:
    print(f"PASS: Email invoice -> invoice_number={email_inv.get('invoice_number')!r}")

# Check 5: All results are dicts (tool_use was always used)
for name, result in stress_results.items():
    if not isinstance(result, dict):
        print(f"FAIL: '{name}' did not return a dict (tool_use not used)")
        all_passed = False

print()
if all_passed:
    print("ALL STRESS TESTS PASSED")
    print("Nullable fields return null, ambiguous data uses escape valves,")
    print("and every response is structured (no text fallback).")
else:
    print("SOME TESTS FAILED -- review the output above.")

---
## Key Takeaways

1. **Prompt-based JSON extraction is unreliable.** The model wraps JSON in markdown, uses inconsistent field names, and represents missing data differently each time.

2. **`tool_choice: "auto"` does not guarantee structured output.** The model may return text instead of a tool call, especially for ambiguous documents.

3. **Required non-nullable fields cause fabrication.** If the schema demands a string value and the document does not contain that information, the model invents data. This is the most dangerous failure mode because it is silent.

4. **The fix is schema design, not prompt engineering:**
   - `"type": ["string", "null"]` for fields that may be absent
   - `"enum": [..., "other"]` for open-ended categories
   - `"enum": [..., "unclear"]` for ambiguous status fields
   - `tool_choice: {"type": "any"}` or `{"type": "tool", "name": "..."}` to guarantee structured output

5. **On the exam:** When a question asks how to prevent extraction hallucination, look for nullable types and enum escape valves -- never "add more instructions to the prompt."

## Next

Lab 1.4: Validation, Retry, and Feedback Loops -- what to do when the extraction has errors.